# 📝 String Master Guide — Sean Edition

## Mental Model

A Python string is an **immutable ribbon of characters**. You can read any position in O(1), but you cannot edit in place — every `s[i] = x` attempt raises TypeError. Concatenation `s + t` copies both ribbons into a new one — O(n). Building a string one char at a time with `+=` in a loop is O(n²). The fix: collect chars in a list, then `''.join(list)` once at the end.

---

## Table of Contents

| # | Section |
|---|---|
| 1 | [Visual Model — Ribbon, Slicing, Two-Pointer, Sliding Window](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [API Reference Table](#3) |
| 4 | [API Demo + O(n²) Trap](#4) |
| 5 | [Pattern 1 — Sliding Window: Longest Substring Without Repeating (LC 3)](#5) |
| 6 | [Pattern 1 — Code](#6) |
| 7 | [Pattern 2 — Two-Pointer Palindrome (LC 125)](#7) |
| 8 | [Pattern 2 — Code](#8) |
| 9 | [Pattern 3 — Anagram / Permutation Sliding Window (LC 567)](#9) |
| 10 | [Pattern 3 — Code](#10) |
| 11 | [Pattern 4 — Encode / Decode Strings (LC 271)](#11) |
| 12 | [Pattern 4 — Code](#12) |
| 13 | [Pattern 5 — Expand Around Center (LC 647 + LC 5)](#13) |
| 14 | [Pattern 5 — Code](#14) |
| 15 | [Decision Map](#15) |
| 16 | [Cheat Sheet — Templates](#16) |
| 17 | [Summary](#17) |

<a id='1'></a>

## 1. Visual Model

### String as Indexed Array

```
s = "abcdef"

 index:  0    1    2    3    4    5
        ┌────┬────┬────┬────┬────┬────┐
        │ a  │ b  │ c  │ d  │ e  │ f  │
        └────┴────┴────┴────┴────┴────┘
neg:    -6   -5   -4   -3   -2   -1

s[2]    → 'c'          O(1)
s[-1]   → 'f'          O(1)
s[1:4]  → 'bcd'        O(j-i) — copies a NEW ribbon
```

### Two-Pointer (Palindrome Check)

```
s = "r a c e c a r"
      L           R    compare s[L] == s[R], then L++, R--
        L       R      compare again
          L   R        compare again
            L=R        done — palindrome!
```

### Sliding Window (Longest Unique Substring)

```
s = "a b c a b c b b"
     0 1 2 3 4 5 6 7

Step 1:  [a b c]       L=0 R=2  window=3  no dupe
Step 2:  [a b c a]     L=0 R=3  'a' dupe! jump L past last 'a'
Step 3:    [b c a]     L=1 R=3  window=3  ok
Step 4:    [b c a b]   L=1 R=4  'b' dupe! jump L past last 'b'
Step 5:      [c a b]   L=2 R=4  window=3  ok
...answer = 3
```

### Why the O(n²) += Trap Hurts

```
BAD — copies grow each iteration:
result = ""
for c in s:           # n iterations
    result += c       # each += copies entire string so far → O(1+2+3+...+n) = O(n²)

GOOD — single allocation at the end:
buf = []
for c in s:           # n iterations
    buf.append(c)     # O(1) amortized
result = ''.join(buf) # one copy → O(n)
```

<a id='2'></a>

## 2. 🔧 Creating / Setup

In [ ]:
from collections import Counter

# --- Literals ---
s = "hello world"
s2 = 'single quotes work too'
s3 = """triple quotes
span multiple lines"""

# --- Slicing ---
# s[start:stop:step] — creates a NEW string, O(j-i)
print(s[0:5])      # 'hello'
print(s[-5:])      # 'world'
print(s[::-1])     # reversed: 'dlrow olleh'

# --- Split / Join ---
words = s.split()          # ['hello', 'world']  split on whitespace
csv = "a,b,c".split(',')   # ['a', 'b', 'c']     split on delimiter
joined = '-'.join(words)   # 'hello-world'        O(n)
print(joined)

# --- Replace ---
print(s.replace('world', 'Python'))  # 'hello Python'  O(n)

# --- f-strings (Python 3.6+) ---
name = "Sean"
print(f"Hello, {name}! Length: {len(name)}")

# --- ord / chr ---
print(ord('a'))   # 97   — char to ASCII
print(chr(97))    # 'a'  — ASCII to char
print(ord('a') - ord('a'))  # 0 — useful for 26-bucket freq arrays

# --- Case / Cleaning ---
t = "  Hello World!  "
print(t.strip())           # 'Hello World!'  removes leading/trailing whitespace
print(t.strip().lower())   # 'hello world!'
print(t.strip().upper())   # 'HELLO WORLD!'

# --- Classification ---
print('a'.isalpha())   # True  — letters only
print('3'.isdigit())   # True  — digits only
print('a3'.isalnum())  # True  — letters or digits (used a lot in palindrome problems)

# --- Counter on string ---
freq = Counter("abracadabra")
print(freq)   # Counter({'a': 5, 'b': 2, 'r': 2, 'c': 1, 'd': 1})

# --- Mutate via list then join ---
# strings are immutable — s[0] = 'x' raises TypeError
mutable = list("hello")  # ['h', 'e', 'l', 'l', 'o']
mutable[0] = 'H'
result = ''.join(mutable)  # 'Hello'
print(result)

<a id='3'></a>

## 3. API Reference

```
OPERATION              COMPLEXITY   WHAT IT DOES
──────────────────────────────────────────────────────
s[i]                   O(1)         Access char at index i
s[i:j]                 O(j-i)       Slice — creates a new string
len(s)                 O(1)         Length
s + t                  O(n+m)       Concatenation — creates new string (avoid in loops!)
''.join(lst)           O(n)         Efficient string build from list
s.split(sep)           O(n)         Split into list
s.replace(old, new)    O(n)         Replace all occurrences
s.lower() / s.upper()  O(n)         Case conversion
s.strip()              O(n)         Remove leading/trailing whitespace
s.find(sub)            O(n*m)       First index of sub, -1 if not found
sub in s               O(n*m)       Substring membership test
s.count(sub)           O(n*m)       Count non-overlapping occurrences
ord(c)                 O(1)         Char to ASCII code
chr(n)                 O(1)         ASCII code to char
sorted(s)              O(n log n)   Returns sorted list of chars
──────────────────────────────────────────────────────
THINGS YOU DO NOT DO:
❌  s[i] = x            — strings are immutable, use list(s) then join
❌  result += char      — O(n²) total in a loop; use list.append + join
❌  s.find() for bool   — use `in` instead
```

In [ ]:
import time

# --- Demo: the += trap vs join fix ---

N = 50_000
chars = ['x'] * N

# BAD: O(n²) — each += allocates a new string
t0 = time.perf_counter()
bad = ""
for c in chars:
    bad += c
t1 = time.perf_counter()
bad_ms = (t1 - t0) * 1000

# GOOD: O(n) — one allocation at the end
t0 = time.perf_counter()
buf = []
for c in chars:
    buf.append(c)
good = ''.join(buf)
t1 = time.perf_counter()
good_ms = (t1 - t0) * 1000

print(f"+=  loop : {bad_ms:.2f} ms   (O(n²))")
print(f"join fix : {good_ms:.2f} ms   (O(n))")
print(f"Speedup  : {bad_ms / good_ms:.1f}x")

# --- Demo: find vs in ---
s = "hello world"
print(s.find('world'))     # 6    — use when you need the index
print('world' in s)        # True — use when you only need existence

# --- Demo: sorted on string ---
print(sorted("dcba"))      # ['a', 'b', 'c', 'd']  — returns a list
print(''.join(sorted("dcba")))  # 'abcd'  — sort-and-join trick for anagram check

<a id='4'></a>

## 4. Decision Map — Pick Your Pattern

```
SIGNAL IN THE PROBLEM                      WHAT TO DO
───────────────────────────────────────────────────────────────
"longest substring without X"              sliding window + seen set/dict
"minimum window containing all chars"      sliding window + two freq maps
"is it a palindrome?"                      two-pointer from ends
"are these anagrams?"                      Counter(s) == Counter(t)
"does s2 contain permutation of s1?"       sliding window freq map
"encode / decode strings"                  length prefix: "4#word"
"expand palindromes"                       expand-around-center, odd+even
"count palindromic substrings"             expand-around-center, count expansions
"reverse words in string"                  split + reverse + join
"is s a rotation of t?"                    s in (t+t)
```

<a id='5'></a>

## 5. Pattern 1 — Sliding Window: Longest Substring Without Repeating

**LC 3 — Medium**

### Problem
Given string `s`, find the length of the longest substring that contains no repeating characters.

### Trick
- Keep a `seen` dict: char → last seen index.
- Right pointer `r` scans forward.
- When `s[r]` was seen before AND its last index is inside the current window, jump `left` past it: `left = seen[s[r]] + 1`.
- Always update `seen[s[r]] = r` and `best = max(best, r - left + 1)`.

### Slow Motion Trace — `s = "abcabcbb"`

```
 r   s[r]  seen                    left  window       best
─────────────────────────────────────────────────────────────
 0    a    {a:0}                    0    [a]           1
 1    b    {a:0, b:1}               0    [a,b]         2
 2    c    {a:0, b:1, c:2}          0    [a,b,c]       3
 3    a    a seen at 0 ≥ left(0)    1    [b,c,a]       3   ← jump left to 1
 4    b    b seen at 1 ≥ left(1)    2    [c,a,b]       3   ← jump left to 2
 5    c    c seen at 2 ≥ left(2)    3    [a,b,c]       3   ← jump left to 3
 6    b    b seen at 4 ≥ left(3)    5    [c,b]         3   ← jump left to 5
 7    b    b seen at 6 ≥ left(5)    7    [b]           3   ← jump left to 7

Answer: 3
```

### Key Insight
Storing **last seen index** (not a boolean) lets you jump `left` in one step instead of shrinking one character at a time. One pass, O(n).

### Complexity
- **Time:** O(n) — each character visited at most twice (added, then jumped past)
- **Space:** O(min(n, |alphabet|)) — at most 128 entries for ASCII

In [ ]:
def length_of_longest_substring(s: str) -> int:
    """
    LC 3 — Longest Substring Without Repeating Characters.

    Sliding window: right pointer expands; when duplicate found,
    left pointer jumps past the previous occurrence in one step.

    Time:  O(n)                  — one pass
    Space: O(min(n, alphabet))   — seen dict, at most 128 entries for ASCII
    """
    seen = {}    # char -> last index where this char was seen
    left = 0     # left edge of current window
    best = 0     # longest window found so far

    for right, ch in enumerate(s):
        # if ch was seen AND that occurrence is inside our window
        # jump left past it to eliminate the duplicate
        if ch in seen and seen[ch] >= left:
            left = seen[ch] + 1   # jump — not shrink one step

        seen[ch] = right                         # always update to latest index
        best = max(best, right - left + 1)       # window size = right - left + 1

    return best


def test_harness(fn, cases):
    # cases: list of (args_tuple, expected)
    passed = 0
    for args, expected in cases:
        result = fn(*args)
        status = "PASS" if result == expected else "FAIL"
        if status == "FAIL":
            print(f"  {status}  input={args}  got={result}  want={expected}")
        passed += (status == "PASS")
    print(f"  {passed}/{len(cases)} passed")


# slow-motion trace case: "abcabcbb" → 3
# each char and its window movement commented inline in the function above

test_harness(length_of_longest_substring, [
    (("abcabcbb",), 3),   # classic case
    (("bbbbb",),    1),   # all same chars
    (("pwwkew",),   3),   # "wke"
    (("",),         0),   # empty string
    (("abcdef",),   6),   # all unique
    (("dvdf",),     3),   # tricky: left jumps past first 'd'
])

print("length_of_longest_substring defined.")

<a id='7'></a>

## 7. Pattern 2 — Two-Pointer Palindrome Check

**LC 125 — Easy**

### Problem
A phrase is a palindrome if, after keeping only alphanumeric characters and ignoring case, it reads the same forward and backward.

### Trick
- Two pointers: `left=0`, `right=len(s)-1`.
- Skip any character that is not alphanumeric (`isalnum()`).
- Compare `s[left].lower()` vs `s[right].lower()`.
- If mismatch → return False. If pointers cross → return True.

### Slow Motion Trace — `s = "A man, a plan, a canal: Panama"`

```
Cleaned (conceptually): "amanaplanacanalpanama"

left  right  s[left]  s[right]  action
──────────────────────────────────────────────
0     29     'A'      'a'       lower both → match, move inward
1     28     'm'      'm'       match, move inward
2     27     'a'      'a'       match, move inward
...   ...    (skip commas, spaces — not isalnum)
...   ...    all pairs match
left > right                    return True
```

### Key Insight
No extra string allocation needed. Skip in place with `isalnum()`. Space O(1).

### Complexity
- **Time:** O(n) — each character visited at most once
- **Space:** O(1) — no auxiliary storage

In [ ]:
def is_palindrome(s: str) -> bool:
    """
    LC 125 — Valid Palindrome.

    Two pointers from both ends, skipping non-alphanumeric characters,
    comparing lowercased chars.

    Time:  O(n)  — single pass
    Space: O(1)  — no extra allocation
    """
    left = 0
    right = len(s) - 1

    while left < right:
        # skip non-alphanumeric from the left
        while left < right and not s[left].isalnum():
            left += 1
        # skip non-alphanumeric from the right
        while left < right and not s[right].isalnum():
            right -= 1

        # compare, ignoring case
        if s[left].lower() != s[right].lower():
            return False

        left += 1
        right -= 1

    return True


test_harness(is_palindrome, [
    (("A man, a plan, a canal: Panama",), True),
    (("race a car",),                     False),
    (("",),                               True),   # empty is palindrome
    ((" ",),                              True),   # only spaces → no alnum → palindrome
    (("0P",),                             False),
    (("Was it a car or a cat I saw?",),    True),
])

print("is_palindrome defined.")

<a id='9'></a>

## 9. Pattern 3 — Anagram / Permutation Sliding Window

**LC 567 — Medium** (also covers LC 242 Valid Anagram)

### Problem
Given strings `s1` and `s2`, return True if `s2` contains any permutation of `s1`.

### Trick
- Build a freq array of size 26 for `s1`.
- Slide a window of size `len(s1)` across `s2`, maintaining a freq array for the window.
- Instead of comparing two 26-element arrays each step (still O(26) = O(1) but messier), track a `match` counter: how many of the 26 buckets are currently balanced.
- When `match == 26`, found a permutation.

### Slow Motion Trace — `s1="ab"`, `s2="eidbaooo"`

```
s1 freq: a=1, b=1  (all others 0)
window size k=2

r=0  add 'e': s2_freq[e]=1  e≠s1 → match stays at 24   window=[e]
     (match counts buckets where s1_freq==s2_freq; initially 24 zero==zero)
r=1  add 'i': s2_freq[i]=1  match=23                    window=[e,i]
     remove 'e' (left=0): s2_freq[e]=0  back to balanced → match=24
r=2  add 'd': match=23      remove 'i': match=24         window=[d]
r=3  add 'b': s2_freq[b]=1==s1_freq[b]=1 → match=25     window=[d,b]
     remove 'd': match=24
r=4  add 'a': s2_freq[a]=1==s1_freq[a]=1 → match=26 ✓   window=[b,a]
     match==26 → return True
```

### Key Insight
The `match` counter lets you update in O(1) per step instead of comparing the full array. You only adjust `match` when a bucket crosses or uncrosses the balance line.

### Complexity
- **Time:** O(n) where n = len(s2)
- **Space:** O(26) = O(1)

In [ ]:
def check_inclusion(s1: str, s2: str) -> bool:
    """
    LC 567 — Permutation in String.

    Sliding window of size len(s1) over s2.
    Uses 26-bucket freq arrays and a match counter for O(1) per-step updates.

    Time:  O(n)   where n = len(s2)
    Space: O(26) = O(1)
    """
    if len(s1) > len(s2):
        return False

    # 26-bucket frequency arrays (index 0 = 'a', index 25 = 'z')
    s1_freq = [0] * 26
    s2_freq = [0] * 26

    k = len(s1)

    # populate s1 freq and the first window of s2
    for i in range(k):
        s1_freq[ord(s1[i]) - ord('a')] += 1
        s2_freq[ord(s2[i]) - ord('a')] += 1

    # count how many of the 26 buckets are already balanced
    match = sum(1 for i in range(26) if s1_freq[i] == s2_freq[i])

    # slide the window across s2
    for right in range(k, len(s2)):
        if match == 26:
            return True

        # add new right character
        idx_in = ord(s2[right]) - ord('a')
        # before adding: if bucket was balanced, it will become unbalanced
        if s2_freq[idx_in] == s1_freq[idx_in]:
            match -= 1
        s2_freq[idx_in] += 1
        # after adding: check if it became balanced
        if s2_freq[idx_in] == s1_freq[idx_in]:
            match += 1

        # remove left character (left = right - k)
        idx_out = ord(s2[right - k]) - ord('a')
        if s2_freq[idx_out] == s1_freq[idx_out]:
            match -= 1
        s2_freq[idx_out] -= 1
        if s2_freq[idx_out] == s1_freq[idx_out]:
            match += 1

    return match == 26  # check the last window


test_harness(check_inclusion, [
    (("ab", "eidbaooo"),  True),   # "ba" at index 3
    (("ab", "eidboaoo"),  False),
    (("adc", "dcda"),     True),   # "cda" is permutation
    (("a", "ab"),         True),
    (("abc", "ab"),       False),  # s1 longer than s2
])

print("check_inclusion defined.")

<a id='11'></a>

## 11. Pattern 4 — Encode / Decode Strings

**LC 271 — Medium**

### Problem
Design an algorithm to serialize a list of strings into a single string and deserialize it back. The strings can contain any character including delimiters.

### Trick: Length Prefix
Encode each word as `"<length>#<word>"`. The `#` is just a separator between the length number and the word. Since we know the exact byte count, we skip exactly that many characters — no ambiguity with any content inside the words.

```
encode(["hello", "#world", "foo#bar"])
→  "5#hello6##world7#foo#bar"
    └──┬──┘ └───┬───┘ └──┬──┘
    len=5  len=6      len=7
```

### Slow Motion Decode — `"5#hello6##world"`

```
i=0: find '#' at position 1  → length=5
     i moves to 2 (after '#')
     grab s[2:7] = "hello"
     i moves to 7

i=7: find '#' at position 8  → length=6
     i moves to 9
     grab s[9:15] = "#world"   ← the # inside is just data
     i moves to 15

Result: ["hello", "#world"]
```

### Key Insight
Length-prefix encoding is separator-free. It does not matter what characters are in the words. This is the same idea used in TCP framing.

### Complexity
- **Encode:** O(n) total characters
- **Decode:** O(n) total characters

In [ ]:
def encode(strs: list[str]) -> str:
    """
    LC 271 — Encode Strings.

    Each word is prefixed with its length and a '#' delimiter:
    ["hello", "world"] → "5#hello5#world"

    Time:  O(n)  total characters
    Space: O(n)
    """
    # build list of chunks, then join once — avoids O(n²) += trap
    parts = []
    for word in strs:
        parts.append(f"{len(word)}#{word}")  # length + sentinel + word
    return ''.join(parts)


def decode(s: str) -> list[str]:
    """
    LC 271 — Decode Strings.

    Scan for '#', read the integer before it as length,
    then grab exactly that many characters after '#'.

    Time:  O(n)  total characters
    Space: O(n)  output list
    """
    result = []
    i = 0

    while i < len(s):
        # find the '#' that separates length from word content
        j = i
        while s[j] != '#':
            j += 1
        # s[i:j] is the length digits, s[j] is '#'
        length = int(s[i:j])
        word_start = j + 1                      # skip the '#'
        result.append(s[word_start:word_start + length])
        i = word_start + length                 # advance past this word

    return result


# round-trip test harness
def test_round_trip(cases):
    passed = 0
    for original in cases:
        encoded = encode(original)
        decoded = decode(encoded)
        status = "PASS" if decoded == original else "FAIL"
        if status == "FAIL":
            print(f"  FAIL  input={original}  encoded={encoded!r}  got={decoded}")
        passed += (status == "PASS")
    print(f"  {passed}/{len(cases)} passed")


test_round_trip([
    ["hello", "world"],
    ["#", "##", "#world"],          # words containing '#'
    ["", "", ""],                    # empty strings
    ["single"],
    ["foo#bar", "baz"],
    [],                              # empty list
])

print("encode/decode defined.")

<a id='13'></a>

## 13. Pattern 5 — Expand Around Center

**LC 647 (Count Palindromic Substrings) + LC 5 (Longest Palindromic Substring)**

### Problem
- LC 647: Count all palindromic substrings in `s`.
- LC 5: Find the longest palindromic substring in `s`.

### Trick
Every palindrome has a center. For a string of length n there are **2n - 1** possible centers:
- n odd-length centers (each character)
- n-1 even-length centers (each gap between characters)

For each center, expand outward while `s[left] == s[right]`. Each successful expansion is one palindrome.

### Slow Motion Trace — `s = "abacaba"`

```
index:  0 1 2 3 4 5 6
chars:  a b a c a b a

Center at index 3 ('c') — ODD:
  expand(3,3): s[3]==s[3]  → "c"       count+1
  expand(2,4): s[2]='a'==s[4]='a' → "aca"  count+1
  expand(1,5): s[1]='b'==s[5]='b' → "bacab" count+1
  expand(0,6): s[0]='a'==s[6]='a' → "abacaba" count+1
  expand(-1,7): out of bounds, stop

Center between index 0 and 1 — EVEN:
  expand(0,1): s[0]='a'≠s[1]='b' → stop immediately (no palindrome)

Center between index 2 and 3 — EVEN:
  expand(2,3): s[2]='a'≠s[3]='c' → stop

Total palindromes for "abacaba": 7 single chars + 3 multi = ... count them all
Answer: 7
```

### Key Insight
2n-1 centers. Each expansion is O(n) worst case. Total O(n²). Space O(1) — no DP table needed.

### Complexity
- **Time:** O(n²)
- **Space:** O(1)

In [ ]:
def _expand(s: str, left: int, right: int) -> int:
    # expand outward from center while chars match
    # returns the count of palindromes found from this center
    count = 0
    while left >= 0 and right < len(s) and s[left] == s[right]:
        count += 1
        left -= 1
        right += 1
    return count


def count_substrings(s: str) -> int:
    """
    LC 647 — Palindromic Substrings.

    For each of the 2n-1 centers, expand outward and count palindromes.

    Time:  O(n²)
    Space: O(1)
    """
    total = 0
    for i in range(len(s)):
        total += _expand(s, i, i)       # odd-length: center is s[i]
        total += _expand(s, i, i + 1)   # even-length: center is gap between s[i] and s[i+1]
    return total


def _expand_range(s: str, left: int, right: int):
    # returns (start, end) indices of the longest palindrome from this center
    while left >= 0 and right < len(s) and s[left] == s[right]:
        left -= 1
        right += 1
    # left and right are now one step past the palindrome boundary
    return left + 1, right - 1


def longest_palindrome(s: str) -> str:
    """
    LC 5 — Longest Palindromic Substring.

    Same expand-around-center approach; track the widest expansion.

    Time:  O(n²)
    Space: O(1)  (excluding output string)
    """
    best_start = 0
    best_end = 0

    for i in range(len(s)):
        # odd-length palindrome
        l, r = _expand_range(s, i, i)
        if r - l > best_end - best_start:
            best_start, best_end = l, r

        # even-length palindrome
        l, r = _expand_range(s, i, i + 1)
        if r - l > best_end - best_start:
            best_start, best_end = l, r

    return s[best_start:best_end + 1]


test_harness(count_substrings, [
    (("abc",),     3),   # 'a','b','c'
    (("aaa",),     6),   # 'a','a','a','aa','aa','aaa'
    (("abacaba",), 7),
    (("",),        0),
    (("a",),       1),
])

test_harness(longest_palindrome, [
    (("babad",),  "bab"),   # "aba" also valid
    (("cbbd",),   "bb"),
    (("a",),      "a"),
    (("racecar",),"racecar"),
    (("abacaba",),"abacaba"),
])

print("palindrome functions defined.")

<a id='15'></a>

## 15. Full Decision Map

```
SIGNAL IN THE PROBLEM                      PATTERN                          COMPLEXITY
──────────────────────────────────────────────────────────────────────────────────────
"longest substring without X"              sliding window + seen dict        O(n) / O(k)
"minimum window containing all chars"      sliding window + two freq maps    O(n) / O(k)
"is it a palindrome?"                      two-pointer from ends             O(n) / O(1)
"are these anagrams?"                      Counter(s) == Counter(t)          O(n) / O(k)
"does s2 contain permutation of s1?"       sliding window + match counter    O(n) / O(1)
"encode / decode strings"                  length prefix: "4#word"           O(n) / O(n)
"count palindromic substrings"             expand-around-center              O(n²) / O(1)
"longest palindromic substring"            expand-around-center              O(n²) / O(1)
"reverse words in string"                  split + reverse + join            O(n) / O(n)
"is s a rotation of t?"                    s in (t+t)                        O(n) / O(n)
"first unique character"                   Counter + scan                    O(n) / O(k)
"longest common prefix"                    zip columns, stop on mismatch     O(n*m) / O(1)
──────────────────────────────────────────────────────────────────────────────────────
k = size of character set (usually 26 or 128)
```

<a id='16'></a>

## 16. Cheat Sheet — Copy-Paste Templates

### Template 1 — Sliding Window (Longest Unique Substring)
```python
seen = {}     # char -> last seen index
left = 0
best = 0
for right, ch in enumerate(s):
    if ch in seen and seen[ch] >= left:
        left = seen[ch] + 1
    seen[ch] = right
    best = max(best, right - left + 1)
return best
```

### Template 2 — Two-Pointer Palindrome
```python
left, right = 0, len(s) - 1
while left < right:
    while left < right and not s[left].isalnum():  left += 1
    while left < right and not s[right].isalnum(): right -= 1
    if s[left].lower() != s[right].lower(): return False
    left += 1; right -= 1
return True
```

### Template 3 — Freq Map Anagram (LC 242)
```python
from collections import Counter
return Counter(s) == Counter(t)
# or with sorted: sorted(s) == sorted(t)  → O(n log n) but simpler
```

### Template 4 — Permutation Sliding Window (LC 567)
```python
s1_freq = [0] * 26
s2_freq = [0] * 26
k = len(s1)
for i in range(k):
    s1_freq[ord(s1[i]) - ord('a')] += 1
    s2_freq[ord(s2[i]) - ord('a')] += 1
match = sum(s1_freq[i] == s2_freq[i] for i in range(26))
for right in range(k, len(s2)):
    if match == 26: return True
    # add right char, remove left char — update match counter each time
    ...
return match == 26
```

### Template 5 — Expand Around Center
```python
def expand(s, l, r):
    while l >= 0 and r < len(s) and s[l] == s[r]:
        l -= 1; r += 1
    return r - l - 1  # palindrome length

best = 0
for i in range(len(s)):
    odd  = expand(s, i, i)
    even = expand(s, i, i + 1)
    best = max(best, odd, even)
```

### Template 6 — Encode / Decode
```python
def encode(strs):
    return ''.join(f"{len(w)}#{w}" for w in strs)

def decode(s):
    res, i = [], 0
    while i < len(s):
        j = s.index('#', i)
        n = int(s[i:j])
        res.append(s[j+1:j+1+n])
        i = j + 1 + n
    return res
```

<a id='17'></a>

## 17. Summary

```
┌─────────────────────────────────────────────────────────────────────┐
│                   STRING MASTER GUIDE — SUMMARY                     │
├──────────────────────────┬──────────────────────────────────────────┤
│ CONCEPT                  │ KEY POINT                                │
├──────────────────────────┼──────────────────────────────────────────┤
│ Immutability             │ s[i]=x raises TypeError; use list+join   │
│ += in loop               │ O(n²) — always use list.append + join    │
│ Slicing s[i:j]           │ Creates a new string — O(j-i) copy       │
├──────────────────────────┼──────────────────────────────────────────┤
│ Sliding Window           │ seen dict + left pointer jump            │
│   LC 3                   │ Longest Substring Without Repeating      │
│   LC 567                 │ Permutation in String (match counter)    │
├──────────────────────────┼──────────────────────────────────────────┤
│ Two-Pointer              │ left+right, skip non-alnum, compare low  │
│   LC 125                 │ Valid Palindrome                         │
├──────────────────────────┼──────────────────────────────────────────┤
│ Freq Map                 │ Counter(s)==Counter(t) or 26-array       │
│   LC 242                 │ Valid Anagram                            │
├──────────────────────────┼──────────────────────────────────────────┤
│ Length Prefix Encoding   │ "4#word" — no delimiter ambiguity        │
│   LC 271                 │ Encode/Decode Strings                    │
├──────────────────────────┼──────────────────────────────────────────┤
│ Expand Around Center     │ 2n-1 centers, odd + even, O(n²)/O(1)    │
│   LC 647                 │ Count Palindromic Substrings             │
│   LC 5                   │ Longest Palindromic Substring            │
└──────────────────────────┴──────────────────────────────────────────┘
```

*End of String Master Guide — Sean Edition*